# American Options Pricing — Longstaff-Schwartz Algorithm

This notebook visualises the **convergence properties** of the LSM algorithm as the underlying
numerical parameters vary. All data is read directly from `csv_output/`, with model parameters
parsed from file names so the notebook stays in sync with whatever run produced the CSVs.

# Convergence Overview

We study how the estimated American put price changes as we vary four numerical parameters:
the path count $N$, the number of exercise dates $M$, the FD time-step count, and the
polynomial degree used in the LSM regression.

## Imports & Configuration

In [232]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [233]:

csv_dir   = "csv_output"
Plots_dir = "plots"
os.makedirs(Plots_dir, exist_ok=True)

### Matplotlib Config

In [ ]:
BLUE = "#74B9FF"
GREEN = "#00CEC9"
RED = "#FD79A8"
ORANGE = "#A29BFE"
PURPLE = "#6C5CE7"
GREY = "#B2BEC3"
DARK_GREY = "#636E72"
GOLD = "#FDCB6E"

plt.rcParams.update({
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.grid": True,
    "grid.color": "#E0E0E0",
    "grid.linewidth": 0.9,
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "legend.framealpha": 0.85,
    "figure.dpi": 120,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

### Load Data

Convergence CSVs follow the naming convention:

```
{type}_convergence_S{S0}_K{K}_r{r}_sig{sigma}_T{T}_{basis}_ord{order}_dates{M}_paths{N}.csv
```

The helpers below parse model parameters directly from the filename so that plots automatically
reflect whichever parameters were used when the C++ binary ran.

`benchmark.csv` — Put prices across spot / vol / maturity scenarios: BS, FD, LSM

`path_convergence_*.csv` — LSM price as path count $N$ varies

`dates_convergence_*.csv` — LSM price as exercise-date count $M$ varies

`fd_convergence_*.csv` — FD price as the number of time steps varies

`order_convergence_*.csv` — LSM price as the polynomial degree varies

`seed_stability_*.csv` — LSM prices across repeated RNG seeds

In [ ]:
def parse_params(filename):
    m = re.search(
        r'S(?P<S0>[\d.]+)_K(?P<K>[\d.]+)_r(?P<r>[\d.]+)_sig(?P<sigma>[\d.]+)_T(?P<T>[\d.]+)',
        filename
    )
    return {k: float(v) for k, v in m.groupdict().items()} if m else {}


def basis_label(filename):
    m = re.search(r'T[\d.]+_(.+?)_ord', os.path.basename(filename))
    return m.group(1) if m else os.path.basename(filename)


def load_convergence(prefix):
    files = sorted(glob.glob(f"{csv_dir}/{prefix}_convergence_*.csv"))
    if not files:
        files = sorted(glob.glob(f"{csv_dir}/{prefix}_*.csv"))
    if not files:
        print(f"No files found for prefix '{prefix}'")
        return []
    return [(basis_label(f), parse_params(f), pd.read_csv(f)) for f in files]


BASIS_COLORS = [PURPLE, RED, GREEN, ORANGE, BLUE]

## Benchmark: Black-Scholes European vs FD American vs LSM American

We compare three pricing methods across a grid of spot prices $S_0$, volatilities $\sigma$,
and maturities $T$ (strike $K$ and rate $r$ are fixed for each scenario).

- **BS European** — closed-form Black-Scholes put price; a lower bound since early exercise premium is excluded
- **FD American** — finite-difference PDE price; the reference for the true American value
- **LSM American** — Monte-Carlo Longstaff-Schwartz price

The **Early Exercise Premium** indicates the additional value from the right to exercise early:

$$\text{EEP} = V^{\text{LSM}} - V^{\text{BS}} \geq 0$$

A negative EEP would indicate LSM underpricing (insufficient paths or basis mismatch).

In [ ]:
benchmark = pd.read_csv(f"{csv_dir}/benchmark.csv")

benchmark["label"] = (
    "$S_0$=" + benchmark["S0"].astype(str)
    + ", $\\sigma$=" + benchmark["Sigma"].astype(str)
    + ", $T$=" + benchmark["T"].astype(str)
)

x     = np.arange(len(benchmark))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, benchmark["BSPrice"],  width, color=BLUE,   label="BS European")
ax.bar(x,         benchmark["FDPrice"],  width, color=ORANGE, label="FD American")
ax.bar(x + width, benchmark["LSMPrice"], width, color=GREEN,  label="LSM American")
ax.set_xticks(x)
ax.set_xticklabels(benchmark["label"], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Option Price")
ax.set_title("Put Option Prices: BS vs FD vs LSM")
ax.legend()
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_benchmark_prices.png", bbox_inches="tight")
plt.show()

fig, ax2 = plt.subplots(figsize=(10, 6))
colors = [GREEN if v >= 0 else RED for v in benchmark["EarlyExPremium"]]
ax2.bar(x, benchmark["EarlyExPremium"], color=colors)
ax2.axhline(0, color=DARK_GREY, linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(benchmark["label"], rotation=35, ha="right", fontsize=8)
ax2.set_ylabel("LSM Price $-$ BS Price")
ax2.set_title("Early Exercise Premium  (green $\\geq 0$, red $< 0$)")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_benchmark_eep.png", bbox_inches="tight")
plt.show()

## Path Count Convergence

The LSM price is a Monte-Carlo estimator so its variance shrinks as the number of simulated
paths $N$ grows. By the Central Limit Theorem the standard error scales as:

$$\text{SE} \propto \frac{1}{\sqrt{N}}$$

The **FD price** (dashed reference line) is plotted alongside the LSM price (left) and
the absolute error $|\hat{V}^{\text{LSM}}_N - V^{\text{FD}}|$ on a log-log scale (right)
to visualise the convergence rate.

In [ ]:
path_data = load_convergence("path")

if path_data:
    _, params, _ = path_data[0]
    suffix = (f"$S_0$={params.get('S0','')}, $K$={params.get('K','')}, "
              f"$r$={params.get('r','')}, $\\sigma$={params.get('sigma','')}, $T$={params.get('T','')}")
else:
    suffix = ""

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(path_data, BASIS_COLORS[2:] + BASIS_COLORS[:2]):
    fd_ref = df["FDPrice"].iloc[-1]
    ax.plot(df["Paths"], df["LSMPrice"], marker="o", ms=4, color=color, label=label)
    ax.axhline(fd_ref, ls="--", lw=1.2, color=color, alpha=0.4)
ax.set_xscale("log")
ax.set_xlabel("Number of Paths ($N$)")
ax.set_ylabel("LSM Price")
ax.set_title(f"LSM Price vs Path Count\n{suffix}")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_paths_price.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(path_data, BASIS_COLORS[2:] + BASIS_COLORS[:2]):
    err = df["Error"].abs()
    ax.plot(df["Paths"], err, marker="o", ms=4, color=color, label=label)
    mask = err > 0
    slope, intercept = np.polyfit(np.log(df["Paths"][mask]), np.log(err[mask]), 1)
    x_fit = df["Paths"].values
    ax.plot(x_fit, np.exp(intercept) * x_fit ** slope,
            ls="--", lw=1.2, color=color, alpha=0.7,
            label=f"fit: $O(N^{{{slope:.2f}}})$")

if path_data:
    _, _, df0 = path_data[0]
    paths = df0["Paths"].values
    err0  = df0["Error"].abs().iloc[0]
    ax.plot(paths, err0 * (paths[0] / paths) ** 0.5,
            ls=":", lw=1.4, color=DARK_GREY, label=r"$O(N^{-1/2})$ theory")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Number of Paths ($N$)")
ax.set_ylabel("|LSM $-$ FD|")
ax.set_title("Absolute Error vs Path Count (log-log)")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_paths_error.png", bbox_inches="tight")
plt.show()

## Exercise-Date Convergence

LSM discretises the continuous-time optimal stopping problem onto a finite grid of $M$
exercise dates. The discrete American price converges to the continuous-time optimum as
$M \to \infty$, with a first-order discretisation error:

$$\left| V^{\text{LSM}}_M - V^{\text{American}} \right| = O\!\left(\frac{1}{M}\right)$$

The **FD reference** (dashed) marks the target price; the right panel shows the absolute error
on a log-log scale to show the convergence rate.

In [ ]:
dates_data = load_convergence("dates")

if dates_data:
    _, params, _ = dates_data[0]
    suffix = (f"$S_0$={params.get('S0','')}, $K$={params.get('K','')}, "
              f"$r$={params.get('r','')}, $\\sigma$={params.get('sigma','')}, $T$={params.get('T','')}")
else:
    suffix = ""

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(dates_data, BASIS_COLORS[3:] + BASIS_COLORS[:3]):
    fd_ref = df["FDPrice"].iloc[-1]
    ax.plot(df["ExerciseDates"], df["LSMPrice"], marker="s", ms=4, color=color, label=label)
    ax.axhline(fd_ref, ls="--", lw=1.2, color=color, alpha=0.4)
ax.set_xscale("log")
ax.set_xlabel("Number of Exercise Dates")
ax.set_ylabel("LSM Price")
ax.set_title(f"LSM Price vs Exercise Dates\n{suffix}")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_dates_price.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(dates_data, BASIS_COLORS[3:] + BASIS_COLORS[:3]):
    err = df["Error"].abs()
    ax.plot(df["ExerciseDates"], err, marker="s", ms=4, color=color, label=label)
    mask = err > 0
    slope, intercept = np.polyfit(np.log(df["ExerciseDates"][mask]), np.log(err[mask]), 1)
    x_fit = df["ExerciseDates"].values
    ax.plot(x_fit, np.exp(intercept) * x_fit ** slope,
            ls="--", lw=1.2, color=color, alpha=0.7,
            label=f"fit: $O(M^{{{slope:.2f}}})$")

if dates_data:
    _, _, df0 = dates_data[0]
    dates = df0["ExerciseDates"].values
    err0  = df0["Error"].abs().iloc[0]
    ax.plot(dates, err0 * (dates[0] / dates) ** 1.0,
            ls=":", lw=1.4, color=DARK_GREY, label=r"$O(M^{-1})$ theory")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Number of Exercise Dates")
ax.set_ylabel("|LSM $-$ FD|")
ax.set_title("Absolute Error vs Exercise Dates (log-log)")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_dates_error.png", bbox_inches="tight")
plt.show()

## Finite-Difference Time-Step Convergence

The finite-difference method solves the Black-Scholes PDE on a discrete time grid.
For a Crank-Nicolson scheme the global truncation error is second-order in time:

$$\left| V^{\text{FD}}_{\Delta t} - V^{\text{American}} \right| = O(\Delta t^2)$$

Increasing the number of time steps reduces the discretisation error, converging the FD
price to the true American value.  The **BS European** price (dashed) provides a
theoretical lower bound.

In [ ]:
fd_data = load_convergence("fd")

if fd_data:
    _, params, _ = fd_data[0]
    suffix = (f"$S_0$={params.get('S0','')}, $K$={params.get('K','')}, "
              f"$r$={params.get('r','')}, $\\sigma$={params.get('sigma','')}, $T$={params.get('T','')}")
else:
    suffix = ""

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(fd_data, BASIS_COLORS[4:] + BASIS_COLORS[:4]):
    bs_ref = df["BSPrice"].iloc[-1]
    ax.plot(df["TimeSteps"], df["FDPrice"], marker="^", ms=4, color=color, label=label)
    ax.axhline(bs_ref, ls="--", lw=1.2, color=color, alpha=0.4, label=f"BS ref ({label})")
ax.set_xscale("log")
ax.set_xlabel("Number of FD Time Steps")
ax.set_ylabel("FD Price")
ax.set_title(f"FD Price vs Time Steps\n{suffix}")
ax.legend(title="Basis", fontsize=8)
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_fd_price.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(fd_data, BASIS_COLORS[4:] + BASIS_COLORS[:4]):
    ax.plot(df["TimeSteps"], df["Error"].abs(), marker="^", ms=4, color=color, label=label)
ax.set_xscale("log")
ax.set_xlabel("Number of FD Time Steps")
ax.set_ylabel("FD $-$ BS (Early Exercise Premium)")
ax.set_title("Early Exercise Premium vs FD Time Steps")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_fd_eep.png", bbox_inches="tight")
plt.show()

## Polynomial Order Convergence

The core of LSM is approximating the **continuation value** $C(t_i)$ via OLS regression.
At each step $t_i$ (working backwards), for ITM paths only, we regress the discounted
future cashflow $e^{-r\,dt} U(t_{i+1})$ onto a polynomial basis:

$$C(t_i) \approx \sum_{j=0}^{p} \hat{\alpha}_j \, \phi_j\!\left(X(t_i)\right)$$

Higher-order polynomials give a richer function space and can reduce approximation bias,
though very high degrees may introduce numerical instability.
We vary the polynomial degree $p$ and record the absolute deviation from the FD reference.

In [ ]:
order_data = load_convergence("order")

if order_data:
    _, params, _ = order_data[0]
    suffix = (f"$S_0$={params.get('S0','')}, $K$={params.get('K','')}, "
              f"$r$={params.get('r','')}, $\\sigma$={params.get('sigma','')}, $T$={params.get('T','')}")
else:
    suffix = ""

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(order_data, BASIS_COLORS[1:] + BASIS_COLORS[:1]):
    fd_ref = df["FDPrice"].iloc[-1]
    ax.plot(df["Order"], df["LSMPrice"], marker="D", ms=5, color=color, label=label)
    ax.axhline(fd_ref, ls="--", lw=1.2, color=color, alpha=0.4)
ax.set_xlabel("Polynomial Order")
ax.set_ylabel("LSM Price")
ax.set_title(f"LSM Price vs Polynomial Order\n{suffix}")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_order_price.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(order_data, BASIS_COLORS[1:] + BASIS_COLORS[:1]):
    ax.plot(df["Order"], df["Error"].abs(), marker="D", ms=5, color=color, label=label)
ax.set_xlabel("Polynomial Order")
ax.set_ylabel("|LSM $-$ FD|")
ax.set_title("Absolute Error vs Polynomial Order")
ax.legend(title="Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_order_error.png", bbox_inches="tight")
plt.show()

## Seed Stability

Monte-Carlo methods are inherently random; different random seeds produce different path
realisations and therefore different price estimates.  For a given basis and path count $N$,
the estimator variance is:

$$\text{Var}\!\left[\hat{V}^{\text{LSM}}\right] \approx \frac{\sigma^2_{\text{payoff}}}{N}$$

The left panel shows individual seed prices with a dashed mean line and a shaded
$\pm 2\sigma$ band (approximately a 95\% confidence interval under normality).
A tight band indicates a stable, low-variance estimator.
The right panel shows the full price distribution per basis as a box-and-whisker.

In [ ]:
seed_data = load_convergence("seed_stability")

box_data   = []
box_labels = []
plot_colors = []

for (label, params, df), color in zip(seed_data, BASIS_COLORS[0:] + BASIS_COLORS[:0]):
    df = df[pd.to_numeric(df["Seed"], errors="coerce").notna()].copy()
    df["Seed"] = df["Seed"].astype(int)
    box_data.append(df["LSMPrice"].values)
    box_labels.append(label)
    plot_colors.append(color)

if seed_data:
    _, params, _ = seed_data[0]
    suffix = (f"$S_0$={params.get('S0','')}, $K$={params.get('K','')}, "
              f"$r$={params.get('r','')}, $\\sigma$={params.get('sigma','')}, $T$={params.get('T','')}")
else:
    suffix = ""

fig, ax = plt.subplots(figsize=(8, 5))
for (label, params, df), color in zip(seed_data, plot_colors):
    df = df[pd.to_numeric(df["Seed"], errors="coerce").notna()].copy()
    df["Seed"] = df["Seed"].astype(int)
    prices = df["LSMPrice"].values
    seeds  = df["Seed"].values
    mean   = prices.mean()
    std    = prices.std()
    ax.axhspan(mean - 2 * std, mean + 2 * std, color=color, alpha=0.12, zorder=1)
    ax.axhline(mean, ls="--", lw=1.4, color=color, alpha=0.8, zorder=2)
    ax.scatter(seeds, prices, color=color, s=30, zorder=3, label=label)
ax.set_xlabel("Random Seed")
ax.set_ylabel("LSM Price")
ax.set_title(f"LSM Prices Across Seeds\n{suffix}")
ax.legend(title=r"Basis  (dashed = mean, band = $\pm 2\sigma$)", fontsize=9)
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_seed_scatter.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(
    box_data,
    labels=box_labels,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker="o", markersize=4, linestyle="none"),
)
for patch, color in zip(bp["boxes"], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for flier, color in zip(bp["fliers"], plot_colors):
    flier.set_markerfacecolor(color)
    flier.set_markeredgecolor(color)
ax.set_ylabel("LSM Price")
ax.set_title("Price Distribution by Basis")
plt.tight_layout()
plt.savefig(f"{Plots_dir}/conv_seed_boxplot.png", bbox_inches="tight")
plt.show()

## Timing

Each convergence run records wall-clock time in milliseconds alongside the price.
Plotting time against the varying parameter reveals the **computational cost** of
each dimension:

- **Paths** $N$ — simulation cost is $O(N)$: doubling paths doubles runtime
- **Exercise dates** $M$ — each path is evaluated at $M$ dates, so cost is $O(M)$
- **FD time steps** — the tridiagonal solve is $O(n_{\text{steps}})$ per spatial grid point
- **Polynomial order** $p$ — OLS cost grows with basis size but is cheap relative to simulation

Reference slopes (dotted) are anchored at the first data point to make the scaling rate visual.

In [ ]:
sections = [
    (path_data,  "Paths",         "Number of Paths ($N$)",        r"$O(N)$ theory", 1.0, BASIS_COLORS[2:] + BASIS_COLORS[:2]),
    (dates_data, "ExerciseDates", "Number of Exercise Dates ($M$)", r"$O(M)$ theory", 1.0, BASIS_COLORS[3:] + BASIS_COLORS[:3]),
    (fd_data,    "TimeSteps",     "Number of FD Time Steps",        r"$O(n)$ theory", 1.0, BASIS_COLORS[4:] + BASIS_COLORS[:4]),
    (order_data, "Order",         "Polynomial Order ($p$)",         None,             None, BASIS_COLORS[1:] + BASIS_COLORS[:1]),
]

for data, x_col, xlabel, ref_label, ref_exp, colors in sections:
    use_loglog = (ref_label is not None)
    fig, ax = plt.subplots(figsize=(8, 5))
    for (label, params, df), color in zip(data, colors):
        x = df[x_col].values
        t = df["Time(ms)"].values
        ax.plot(x, t, marker="o", ms=4, color=color, label=label)
        if use_loglog:
            slope, intercept = np.polyfit(np.log(x), np.log(t), 1)
            ax.plot(x, np.exp(intercept) * x ** slope,
                    ls="--", lw=1.2, color=color, alpha=0.7,
                    label=f"fit: $O(n^{{{slope:.2f}}})$")
            ax.plot(x, t[0] * (x / x[0]) ** ref_exp,
                    ls=":", lw=1.4, color=DARK_GREY, label=ref_label)
    if use_loglog:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Wall-clock time (ms)")
    ax.set_title(f"Time vs {x_col}")
    ax.legend(title="Basis", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{Plots_dir}/conv_timing_{x_col.lower()}.png", bbox_inches="tight")
    plt.show()

----
The styling and formatting of the plots are inspired by the LSM pricer implementation by [LSM Pricer, Dialid Santiago](https://quantgirluk.github.io/Understanding-Quantitative-Finance/AOS.html)